In [1]:
# Install required packages
!pip install pandas requests beautifulsoup4 lxml openpyxl

In [2]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# Set display options for better viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print("Libraries imported successfully")
print(f"Current Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Libraries imported successfully
Current Date: 2026-05-02 02:09


In [3]:
# Load all completed match results from Premier League 2025/26
# Data source: FBref Scores & Fixtures (Matchweeks 1-29)

print("Loading match results data...")

# Create match results dataset
match_results = """
Wk,Date,Home,Score_Home,Score_Away,Away
1,2025-08-15,Liverpool,4,2,Bournemouth
1,2025-08-16,Aston Villa,0,0,Newcastle United
1,2025-08-16,Sunderland,3,0,West Ham United
1,2025-08-16,Brighton,1,1,Fulham
1,2025-08-16,Tottenham Hotspur,3,0,Burnley
1,2025-08-16,Wolves,0,4,Manchester City
1,2025-08-17,Chelsea,0,0,Crystal Palace
1,2025-08-17,Nottingham Forest,3,1,Brentford
1,2025-08-17,Manchester Utd,0,1,Arsenal
1,2025-08-18,Leeds United,1,0,Everton
2,2025-08-22,West Ham United,1,5,Chelsea
2,2025-08-23,Manchester City,0,2,Tottenham Hotspur
2,2025-08-23,Bournemouth,1,0,Wolves
2,2025-08-23,Brentford,1,0,Aston Villa
2,2025-08-23,Burnley,2,0,Sunderland
2,2025-08-23,Arsenal,5,0,Leeds United
2,2025-08-24,Crystal Palace,1,1,Nottingham Forest
2,2025-08-24,Everton,2,0,Brighton
2,2025-08-24,Fulham,1,1,Manchester Utd
2,2025-08-25,Newcastle United,2,3,Liverpool
"""

# Convert to DataFrame
from io import StringIO
matches_df = pd.read_csv(StringIO(match_results))

print(f"Loaded {len(matches_df)} matches")
print(f"Matchweeks: {matches_df['Wk'].min()} to {matches_df['Wk'].max()}")
print("\nFirst 10 matches:")
matches_df.head(10)

Loading match results data...
Loaded 20 matches
Matchweeks: 1 to 2

First 10 matches:


,Wk,Date,Home,Score_Home,Score_Away,Away
0,1,2025-08-15,Liverpool,4,2,Bournemouth
1,1,2025-08-16,Aston Villa,0,0,Newcastle United
2,1,2025-08-16,Sunderland,3,0,West Ham United
3,1,2025-08-16,Brighton,1,1,Fulham
4,1,2025-08-16,Tottenham Hotspur,3,0,Burnley
5,1,2025-08-16,Wolves,0,4,Manchester City
6,1,2025-08-17,Chelsea,0,0,Crystal Palace
7,1,2025-08-17,Nottingham Forest,3,1,Brentford
8,1,2025-08-17,Manchester Utd,0,1,Arsenal
9,1,2025-08-18,Leeds United,1,0,Everton


In [4]:
# Load COMPLETE match results for all 29 matchweeks
print("Loading complete match results (290 matches)...")

# Since we have all match data, we'll load it efficiently
# In real projects, this data would come from: CSV file, database, or API

# For now, we'll create a function to process match results
def create_match_dataframe(match_list):
    """Convert match results into structured DataFrame"""
    return pd.DataFrame(match_list)

# Sample data structure - in next cell we'll add all 290 matches
# For now, let's process what we have and show the calculation method

print(f" Currently loaded: {len(matches_df)} matches")
print("\nNext step: We'll calculate team statistics from these results")
print("This demonstrates Python's data processing capabilities\n")

# Show data structure
print("Match data structure:")
matches_df.info()

Loading complete match results (290 matches)...
 Currently loaded: 20 matches

Next step: We'll calculate team statistics from these results
This demonstrates Python's data processing capabilities

Match data structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Wk          20 non-null     int64 
 1   Date        20 non-null     object
 2   Home        20 non-null     object
 3   Score_Home  20 non-null     int64 
 4   Score_Away  20 non-null     int64 
 5   Away        20 non-null     object
dtypes: int64(3), object(3)
memory usage: 1.1+ KB


In [5]:
# Calculate points and statistics from match results
print("Calculating team statistics from match results...\n")

# Initialize team statistics dictionary
teams = set(matches_df['Home'].unique()) | set(matches_df['Away'].unique())
team_stats = {team: {'MP': 0, 'W': 0, 'D': 0, 'L': 0, 'GF': 0, 'GA': 0, 'Pts': 0} 
              for team in teams}

# Process each match
for idx, match in matches_df.iterrows():
    home_team = match['Home']
    away_team = match['Away']
    home_goals = match['Score_Home']
    away_goals = match['Score_Away']
    
    # Update matches played
    team_stats[home_team]['MP'] += 1
    team_stats[away_team]['MP'] += 1
    
    # Update goals
    team_stats[home_team]['GF'] += home_goals
    team_stats[home_team]['GA'] += away_goals
    team_stats[away_team]['GF'] += away_goals
    team_stats[away_team]['GA'] += home_goals
    
    # Determine result and award points
    if home_goals > away_goals:  # Home win
        team_stats[home_team]['W'] += 1
        team_stats[home_team]['Pts'] += 3
        team_stats[away_team]['L'] += 1
    elif home_goals < away_goals:  # Away win
        team_stats[away_team]['W'] += 1
        team_stats[away_team]['Pts'] += 3
        team_stats[home_team]['L'] += 1
    else:  # Draw
        team_stats[home_team]['D'] += 1
        team_stats[home_team]['Pts'] += 1
        team_stats[away_team]['D'] += 1
        team_stats[away_team]['Pts'] += 1

# Convert to DataFrame
calculated_standings = pd.DataFrame.from_dict(team_stats, orient='index')
calculated_standings = calculated_standings.reset_index().rename(columns={'index': 'Team'})

# Calculate GD
calculated_standings['GD'] = calculated_standings['GF'] - calculated_standings['GA']

# Sort by points, then GD
calculated_standings = calculated_standings.sort_values(['Pts', 'GD'], ascending=False)
calculated_standings['Position'] = range(1, len(calculated_standings) + 1)

# Reorder columns
calculated_standings = calculated_standings[['Position', 'Team', 'MP', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts']]

print(f"Calculated standings from {len(matches_df)} matches")
print(f"Processed {len(calculated_standings)} teams\n")
print("Calculated Standings (from first 20 matches only):")
calculated_standings

Calculating team statistics from match results...

Calculated standings from 20 matches
Processed 20 teams

Calculated Standings (from first 20 matches only):


,Position,Team,MP,W,D,L,GF,GA,GD,Pts
19,1,Arsenal,2,2,0,0,6,0,6,6
8,2,Tottenham Hotspur,2,2,0,0,5,0,5,6
0,3,Liverpool,2,2,0,0,7,4,3,6
16,4,Chelsea,2,1,1,0,5,1,4,4
6,5,Nottingham Forest,2,1,1,0,4,2,2,4
11,6,Manchester City,2,1,0,1,4,2,2,3
12,7,Sunderland,2,1,0,1,3,2,1,3
14,8,Everton,2,1,0,1,2,1,1,3
7,9,Burnley,2,1,0,1,2,3,-1,3
17,10,Bournemouth,2,1,0,1,3,4,-1,3


In [6]:
# Create CSV file with ALL 290 match results
print("Creating complete match results CSV file...")

# I'll show you how to structure the data
# You can paste your complete match results here

# For now, let's create the file structure
import csv

# Define filename
csv_filename = 'PL_Match_Results_2025_26.csv'

print(f"Ready to create: {csv_filename}")
print("\nNext: We'll load your complete match data from the CSV file")
print("This demonstrates professional data workflow: CSV → Python → Analysis")

Creating complete match results CSV file...
Ready to create: PL_Match_Results_2025_26.csv

Next: We'll load your complete match data from the CSV file
This demonstrates professional data workflow: CSV → Python → Analysis


In [7]:
# Load match results using the correct filename
print("Loading complete match results...")

# Use the full file path
file_path = r"C:\Users\hanis\Desktop\Projects\PL winner prediction\PL_Match_Results_2025_26.csv.txt"

matches_df = pd.read_csv(file_path)

print(f"Successfully loaded {len(matches_df)} matches")
print(f"Matchweeks: {matches_df['Wk'].min()} to {matches_df['Wk'].max()}")
print(f"Date range: {matches_df['Date'].min()} to {matches_df['Date'].max()}\n")

print("First 5 matches:")
print(matches_df.head())
print("\nLast 5 matches:")
print(matches_df.tail())

Loading complete match results...
Successfully loaded 291 matches
Matchweeks: 1 to 31
Date range: 2025-08-15 to 2026-03-05

First 5 matches:
   Wk        Date               Home  Score_Home  Score_Away              Away
0   1  2025-08-15          Liverpool           4           2       Bournemouth
1   1  2025-08-16        Aston Villa           0           0  Newcastle United
2   1  2025-08-16         Sunderland           3           0   West Ham United
3   1  2025-08-16           Brighton           1           1            Fulham
4   1  2025-08-16  Tottenham Hotspur           3           0           Burnley

Last 5 matches:
     Wk        Date               Home  Score_Home  Score_Away  \
286  29  2026-03-04             Fulham           0           1   
287  29  2026-03-04        Aston Villa           1           4   
288  29  2026-03-04   Newcastle United           2           1   
289  29  2026-03-05  Tottenham Hotspur           1           3   
290  31  2026-02-18             Wolves

In [8]:
# Calculate Premier League standings from all match results
print("Calculating team statistics from 291 matches...\n")

# Get all unique teams
all_teams = set(matches_df['Home'].unique()) | set(matches_df['Away'].unique())
print(f"Teams found: {len(all_teams)}")

# Initialize statistics for each team
team_stats = {team: {
    'MP': 0, 'W': 0, 'D': 0, 'L': 0, 
    'GF': 0, 'GA': 0, 'GD': 0, 'Pts': 0,
    'Home_W': 0, 'Home_D': 0, 'Home_L': 0,
    'Away_W': 0, 'Away_D': 0, 'Away_L': 0
} for team in all_teams}

# Process each match
for idx, match in matches_df.iterrows():
    home_team = match['Home']
    away_team = match['Away']
    home_goals = match['Score_Home']
    away_goals = match['Score_Away']
    
    # Update matches played
    team_stats[home_team]['MP'] += 1
    team_stats[away_team]['MP'] += 1
    
    # Update goals
    team_stats[home_team]['GF'] += home_goals
    team_stats[home_team]['GA'] += away_goals
    team_stats[away_team]['GF'] += away_goals
    team_stats[away_team]['GA'] += home_goals
    
    # Determine result and update stats
    if home_goals > away_goals:  # Home win
        team_stats[home_team]['W'] += 1
        team_stats[home_team]['Home_W'] += 1
        team_stats[home_team]['Pts'] += 3
        team_stats[away_team]['L'] += 1
        team_stats[away_team]['Away_L'] += 1
        
    elif home_goals < away_goals:  # Away win
        team_stats[away_team]['W'] += 1
        team_stats[away_team]['Away_W'] += 1
        team_stats[away_team]['Pts'] += 3
        team_stats[home_team]['L'] += 1
        team_stats[home_team]['Home_L'] += 1
        
    else:  # Draw
        team_stats[home_team]['D'] += 1
        team_stats[home_team]['Home_D'] += 1
        team_stats[home_team]['Pts'] += 1
        team_stats[away_team]['D'] += 1
        team_stats[away_team]['Away_D'] += 1
        team_stats[away_team]['Pts'] += 1

# Convert to DataFrame
calculated_standings = pd.DataFrame.from_dict(team_stats, orient='index')
calculated_standings = calculated_standings.reset_index().rename(columns={'index': 'Team'})

# Calculate goal difference
calculated_standings['GD'] = calculated_standings['GF'] - calculated_standings['GA']

# Calculate Points Per Game
calculated_standings['PPG'] = (calculated_standings['Pts'] / calculated_standings['MP']).round(2)

# Sort by Points (descending), then GD (descending)
calculated_standings = calculated_standings.sort_values(['Pts', 'GD'], ascending=[False, False])

# Add position
calculated_standings.insert(0, 'Position', range(1, len(calculated_standings) + 1))

# Select columns to display
display_cols = ['Position', 'Team', 'MP', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'PPG']
final_standings = calculated_standings[display_cols]

print(" Standings calculated successfully!\n")
print("=" * 80)
print("PREMIER LEAGUE 2025/26 STANDINGS (Calculated from Match Results)")
print("=" * 80)
print(final_standings.to_string(index=False))

# Save for later use
print("\n Standings saved as 'final_standings' variable")

Calculating team statistics from 291 matches...

Teams found: 20
 Standings calculated successfully!

PREMIER LEAGUE 2025/26 STANDINGS (Calculated from Match Results)
 Position              Team  MP  W  D  L  GF  GA  GD  Pts  PPG
        1           Arsenal  30 20  7  3  59  22  37   67 2.23
        2   Manchester City  29 18  6  5  59  27  32   60 2.07
        3    Manchester Utd  29 14  9  6  51  40  11   51 1.76
        4       Aston Villa  29 15  6  8  39  34   5   51 1.76
        5           Chelsea  29 13  9  7  53  34  19   48 1.66
        6         Liverpool  29 14  6  9  48  39   9   48 1.66
        7         Brentford  29 13  5 11  44  40   4   44 1.52
        8           Everton  29 12  7 10  34  33   1   43 1.48
        9       Bournemouth  29  9 13  7  44  46  -2   40 1.38
       10            Fulham  29 12  4 13  40  43  -3   40 1.38
       11        Sunderland  29 10 10  9  30  34  -4   40 1.38
       12  Newcastle United  29 11  6 12  42  43  -1   39 1.34
       13    C

In [9]:
# Calculate Performance Consistency Metrics (H1)
print("Calculating Performance Consistency Metrics...\n")

# For each team, calculate match-by-match points and variance
team_consistency = {}

for team in all_teams:
    # Get all matches for this team
    team_matches = matches_df[
        (matches_df['Home'] == team) | (matches_df['Away'] == team)
    ].copy()
    
    # Calculate points won in each match
    points_per_match = []
    
    for idx, match in team_matches.iterrows():
        home_goals = match['Score_Home']
        away_goals = match['Score_Away']
        
        if match['Home'] == team:  # Team playing at home
            if home_goals > away_goals:
                points_per_match.append(3)
            elif home_goals == away_goals:
                points_per_match.append(1)
            else:
                points_per_match.append(0)
        else:  # Team playing away
            if away_goals > home_goals:
                points_per_match.append(3)
            elif home_goals == away_goals:
                points_per_match.append(1)
            else:
                points_per_match.append(0)
    
    # Calculate consistency metrics
    team_consistency[team] = {
        'Points_Variance': np.var(points_per_match),
        'Points_StdDev': np.std(points_per_match),
        'Consistency_Score': 1 / (np.std(points_per_match) + 0.1)  # Higher = more consistent
    }

# Convert to DataFrame
consistency_df = pd.DataFrame.from_dict(team_consistency, orient='index')
consistency_df = consistency_df.reset_index().rename(columns={'index': 'Team'})

# Merge with standings
standings_with_consistency = final_standings.merge(consistency_df, on='Team')

# Sort by consistency score (most consistent teams first)
print("=" * 80)
print("PERFORMANCE CONSISTENCY ANALYSIS")
print("=" * 80)
print("\nMost Consistent Teams (Low Variance = Steady Performance):")
most_consistent = standings_with_consistency.sort_values('Points_StdDev').head(10)
print(most_consistent[['Position', 'Team', 'Pts', 'Points_StdDev', 'Consistency_Score']].to_string(index=False))

print("\n\nMost Volatile Teams (High Variance = Unpredictable):")
most_volatile = standings_with_consistency.sort_values('Points_StdDev', ascending=False).head(10)
print(most_volatile[['Position', 'Team', 'Pts', 'Points_StdDev', 'Consistency_Score']].to_string(index=False))

# Save for later
print("\n Consistency metrics saved")

Calculating Performance Consistency Metrics...

PERFORMANCE CONSISTENCY ANALYSIS

Most Consistent Teams (Low Variance = Steady Performance):
 Position              Team  Pts  Points_StdDev  Consistency_Score
       20            Wolves   16       0.921352           0.979095
       19           Burnley   19       1.026405           0.887780
        1           Arsenal   67       1.116045           0.822338
        9       Bournemouth   40       1.157101           0.795481
       15      Leeds United   31       1.172414           0.785908
       16 Tottenham Hotspur   29       1.203443           0.767199
       18   West Ham United   28       1.217197           0.759188
       17 Nottingham Forest   28       1.217197           0.759188
       14          Brighton   37       1.228864           0.752522
        2   Manchester City   60       1.229831           0.751975


Most Volatile Teams (High Variance = Unpredictable):
 Position             Team  Pts  Points_StdDev  Consistency_Score
 

In [10]:
# Calculate Last 5 Games Form (H5 - Momentum)
print("Calculating Last 5 Games Form & Momentum...\n")

team_form = {}

for team in all_teams:
    # Get all matches for this team, sorted by date
    team_matches = matches_df[
        (matches_df['Home'] == team) | (matches_df['Away'] == team)
    ].copy().sort_values('Date')
    
    # Get last 5 matches
    last_5 = team_matches.tail(5)
    
    # Calculate points from last 5
    last_5_points = 0
    results_string = []
    
    for idx, match in last_5.iterrows():
        home_goals = match['Score_Home']
        away_goals = match['Score_Away']
        
        if match['Home'] == team:  # Team at home
            if home_goals > away_goals:
                last_5_points += 3
                results_string.append('W')
            elif home_goals == away_goals:
                last_5_points += 1
                results_string.append('D')
            else:
                results_string.append('L')
        else:  # Team away
            if away_goals > home_goals:
                last_5_points += 3
                results_string.append('W')
            elif home_goals == away_goals:
                last_5_points += 1
                results_string.append('D')
            else:
                results_string.append('L')
    
    team_form[team] = {
        'Last_5_Results': '-'.join(results_string),
        'Last_5_Points': last_5_points,
        'Last_5_PPG': round(last_5_points / 5, 2),
        'Form_Trend': 'Improving' if last_5_points >= 10 else ('Declining' if last_5_points <= 4 else 'Stable')
    }

# Convert to DataFrame
form_df = pd.DataFrame.from_dict(team_form, orient='index')
form_df = form_df.reset_index().rename(columns={'index': 'Team'})

# Merge with standings
standings_with_form = final_standings.merge(form_df, on='Team')

print("=" * 80)
print("LAST 5 GAMES FORM & MOMENTUM ANALYSIS")
print("=" * 80)

print("\n BEST FORM (Last 5 Games):")
best_form = standings_with_form.sort_values('Last_5_Points', ascending=False).head(10)
print(best_form[['Position', 'Team', 'Pts', 'Last_5_Results', 'Last_5_Points', 'Last_5_PPG', 'Form_Trend']].to_string(index=False))

print("\n\n WORST FORM (Last 5 Games):")
worst_form = standings_with_form.sort_values('Last_5_Points').head(10)
print(worst_form[['Position', 'Team', 'Pts', 'Last_5_Results', 'Last_5_Points', 'Last_5_PPG', 'Form_Trend']].to_string(index=False))

print("\n Form and momentum metrics calculated")

Calculating Last 5 Games Form & Momentum...

LAST 5 GAMES FORM & MOMENTUM ANALYSIS

 BEST FORM (Last 5 Games):
 Position            Team  Pts Last_5_Results  Last_5_Points  Last_5_PPG Form_Trend
        2 Manchester City   60      W-W-W-W-D             13         2.6  Improving
        1         Arsenal   67      D-D-W-W-W             11         2.2  Improving
        3  Manchester Utd   51      W-D-W-W-L             10         2.0  Improving
        8         Everton   43      W-L-L-W-W              9         1.8     Stable
       13  Crystal Palace   38      W-L-W-L-W              9         1.8     Stable
        6       Liverpool   48      L-W-W-W-L              9         1.8     Stable
       18 West Ham United   28      W-D-D-L-W              8         1.6     Stable
       20          Wolves   16      D-D-L-W-W              8         1.6     Stable
        7       Brentford   44      W-D-L-W-D              8         1.6     Stable
        5         Chelsea   48      W-D-D-L-W    

In [11]:
# CHAMPIONSHIP PREDICTION MODEL
print("=" * 80)
print("PREMIER LEAGUE 2025/26 CHAMPIONSHIP PREDICTION")
print("=" * 80)

# Combine all metrics
final_analysis = final_standings.merge(consistency_df, on='Team').merge(form_df, on='Team')

# Focus on top 6 contenders
top_6 = final_analysis.head(6)

print("\n TOP 6 CONTENDERS - COMPLETE ANALYSIS:\n")
display_cols = ['Position', 'Team', 'MP', 'Pts', 'PPG', 'Points_StdDev', 
                'Last_5_Points', 'Last_5_PPG', 'Form_Trend']
print(top_6[display_cols].to_string(index=False))

# Calculate Championship Score (weighted model)
print("\n\n CHAMPIONSHIP PREDICTION MODEL:")
print("-" * 80)

for idx, team_row in top_6.iterrows():
    team = team_row['Team']
    
    # Weighted factors
    current_position_score = (7 - team_row['Position']) / 6  # Higher position = higher score
    consistency_score = team_row['Consistency_Score'] / 1.0  # Normalize
    form_score = team_row['Last_5_PPG'] / 3.0  # Normalize (max is 3.0)
    ppg_score = team_row['PPG'] / 3.0  # Normalize
    
    # Weighted Championship Score
    championship_score = (
        0.40 * current_position_score +  # Current position (40%)
        0.25 * form_score +               # Recent form (25%)
        0.20 * ppg_score +                # Overall PPG (20%)
        0.15 * consistency_score          # Consistency (15%)
    )
    
    # Games remaining
    games_remaining = 38 - team_row['MP']
    
    # Predicted final points
    predicted_final_points = team_row['Pts'] + (games_remaining * team_row['Last_5_PPG'])
    
    print(f"\n{team}:")
    print(f"  Current: {team_row['Pts']} pts ({team_row['Position']}st) | PPG: {team_row['PPG']}")
    print(f"  Form: {team_row['Last_5_Results']} ({team_row['Last_5_PPG']} PPG)")
    print(f"  Consistency: {team_row['Consistency_Score']:.3f}")
    print(f"  Games Remaining: {games_remaining}")
    print(f"  Predicted Final Points: {predicted_final_points:.0f}")
    print(f"  Championship Score: {championship_score:.3f}")

# Determine winner
print("\n" + "=" * 80)
print(" FINAL PREDICTION:")
print("=" * 80)

# Calculate for all top 6
predictions = []
for idx, team_row in top_6.iterrows():
    games_remaining = 38 - team_row['MP']
    predicted_points = team_row['Pts'] + (games_remaining * team_row['Last_5_PPG'])
    predictions.append({
        'Team': team_row['Team'],
        'Current_Pts': team_row['Pts'],
        'Predicted_Final_Pts': predicted_points,
        'Last_5_PPG': team_row['Last_5_PPG']
    })

pred_df = pd.DataFrame(predictions).sort_values('Predicted_Final_Pts', ascending=False)
print("\nPredicted Final Standings (Top 6):")
print(pred_df.to_string(index=False))

winner = pred_df.iloc[0]
print(f"\n PREDICTED CHAMPION: {winner['Team']}")
print(f"   Predicted Final Points: {winner['Predicted_Final_Pts']:.0f}")
print(f"   Current Form: {winner['Last_5_PPG']} PPG (last 5 games)")
print(f"\n Confidence: HIGH (based on current form and consistency)")

PREMIER LEAGUE 2025/26 CHAMPIONSHIP PREDICTION

 TOP 6 CONTENDERS - COMPLETE ANALYSIS:

 Position            Team  MP  Pts  PPG  Points_StdDev  Last_5_Points  Last_5_PPG Form_Trend
        1         Arsenal  30   67 2.23       1.116045             11         2.2  Improving
        2 Manchester City  29   60 2.07       1.229831             13         2.6  Improving
        3  Manchester Utd  29   51 1.76       1.249970             10         2.0  Improving
        4     Aston Villa  29   51 1.76       1.330159              5         1.0     Stable
        5         Chelsea  29   48 1.66       1.266977              8         1.6     Stable
        6       Liverpool  29   48 1.66       1.346153              9         1.8     Stable


 CHAMPIONSHIP PREDICTION MODEL:
--------------------------------------------------------------------------------

Arsenal:
  Current: 67 pts (1st) | PPG: 2.23
  Form: D-D-W-W-W (2.2 PPG)
  Consistency: 0.822
  Games Remaining: 8
  Predicted Final Points: 85
 

In [12]:
import os
# Create Excel writer
output_file = 'PL_Analysis_Processed_Data.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    
    # Sheet 1: Final Standings
    final_standings.to_excel(writer, sheet_name='Standings', index=False)
    print("- Saved: Standings")
    
    # Sheet 2: Consistency Analysis
    standings_with_consistency.to_excel(writer, sheet_name='Consistency_Analysis', index=False)
    print("- Saved: Consistency Analysis")
    
    # Sheet 3: Form Analysis
    standings_with_form.to_excel(writer, sheet_name='Form_Analysis', index=False)
    print("- Saved: Form Analysis")
    
    # Sheet 4: Complete Analysis (all metrics combined)
    final_analysis.to_excel(writer, sheet_name='Complete_Analysis', index=False)
    print("- Saved: Complete Analysis")
    
    # Sheet 5: Match Results
    matches_df.to_excel(writer, sheet_name='Match_Results', index=False)
    print("- Saved: Match Results")


print("DATA PROCESSING SUMMARY")
print("=" * 80)
print(f"Total Matches Processed: {len(matches_df)}")
print(f"Teams Analyzed: {len(final_standings)}")
print(f"Matchweeks Completed: {matches_df['Wk'].max()}")
print(f"\nPredicted Champion: Arsenal (85 points)")
print(f"Runner-up: Manchester City (83 points)")

- Saved: Standings
- Saved: Consistency Analysis
- Saved: Form Analysis
- Saved: Complete Analysis
- Saved: Match Results
DATA PROCESSING SUMMARY
Total Matches Processed: 291
Teams Analyzed: 20
Matchweeks Completed: 31

Predicted Champion: Arsenal (85 points)
Runner-up: Manchester City (83 points)
